# Seed MySQL Database from Volume

This notebook loads initial test data from Databricks Volume into MySQL for development and testing.

## Source Data

**Volume Path**: `/Volumes/smart_claims_dev/00_landing/claimsdb`

**Files**:
* `policies.csv` → `policy` table
* `claims.csv` → `claim` table
* `customers.csv` → `customer` table

## Target

**MySQL**: localhost (127.0.0.1:3306)
**Database**: claims_dev

## Process

1. Read CSV files from volume
2. Apply column transformations
3. Write to MySQL via JDBC
4. Verify data loaded correctly

**Note**: This is a one-time seed operation. After this, use the incremental ingestion notebook for ongoing updates.

In [0]:
# Load configuration from query parameters
config = {
    'mysql_host': dbutils.widgets.get('mysql_host'),
    'mysql_port': dbutils.widgets.get('mysql_port'),
    'mysql_database': dbutils.widgets.get('mysql_database'),
    'mysql_user': dbutils.widgets.get('mysql_user'),
    'mysql_password': dbutils.widgets.get('mysql_password'),
    'source_volume_path': dbutils.widgets.get('source_volume_path')
}

# Build JDBC URL
jdbc_url = f"jdbc:mysql://{config['mysql_host']}:{config['mysql_port']}/{config['mysql_database']}"

# Display configuration
print("MySQL Configuration:")
print(f"  Host: {config['mysql_host']}")
print(f"  Port: {config['mysql_port']}")
print(f"  Database: {config['mysql_database']}")
print(f"  User: {config['mysql_user']}")
print(f"  Password: {'*' * 8}")
print(f"  JDBC URL: {jdbc_url}")
print(f"\nSource:")
print(f"  Volume Path: {config['source_volume_path']}")

print("\n✓ Configuration loaded")

In [0]:
# JDBC connection properties for MySQL
jdbc_properties = {
    "user": config['mysql_user'],
    "password": config['mysql_password'],
    "driver": "com.mysql.cj.jdbc.Driver",
    "serverTimezone": "UTC",
    "useSSL": "false",
    "allowPublicKeyRetrieval": "true",
    "rewriteBatchedStatements": "true"  # Improve batch insert performance
}

print("✓ JDBC properties configured")
print("  Driver: MySQL Connector/J")
print("  Batch mode: Enabled for better performance")

In [0]:
# List available CSV files in the volume
import os

try:
    files = dbutils.fs.ls(config['source_volume_path'])
    
    print(f"Files in {config['source_volume_path']}:\n")
    print("=" * 80)
    
    csv_files = [f for f in files if f.name.endswith('.csv')]
    
    if len(csv_files) == 0:
        print("❌ No CSV files found!")
    else:
        for file in csv_files:
            size_mb = file.size / (1024 * 1024)
            print(f"  ✓ {file.name:<30} {size_mb:>10.2f} MB")
        
        print("=" * 80)
        print(f"\nTotal CSV files: {len(csv_files)}")
        
        # Check for required files
        required_files = ['policies.csv', 'claims.csv', 'customers.csv']
        file_names = [f.name for f in csv_files]
        
        missing = [f for f in required_files if f not in file_names]
        if missing:
            print(f"\n⚠️  Missing required files: {', '.join(missing)}")
        else:
            print(f"\n✓ All required files present")
            
except Exception as e:
    print(f"❌ Error listing files: {e}")
    print("\nMake sure the volume path is correct and accessible.")
    raise

## Step 1: Read and Transform CSV Files

Each CSV file requires specific transformations:

### policies.csv → policy table
* Rename all UPPERCASE columns to lowercase
* Direct mapping to policy table schema

### claims.csv → claim table
* Rename: `age` → `driver_age`
* Rename: `date` → `incident_date`
* Rename: `hour` → `incident_hour`
* Rename: `type` → `incident_type`
* Rename: `severity` → `incident_severity`

### customers.csv → customer table
* Cast `customer_id` to INT type

In [0]:
from pyspark.sql.functions import col, lower, trim

# Read policies.csv
policies_path = f"{config['source_volume_path']}/policies.csv"

print("Reading policies.csv...")
policies_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(policies_path)

print(f"  Rows: {policies_df.count():,}")
print(f"\nOriginal columns:")
for col_name in policies_df.columns:
    print(f"    {col_name}")

# Transform: Rename all columns to lowercase
policies_transformed = policies_df
for col_name in policies_df.columns:
    policies_transformed = policies_transformed.withColumnRenamed(col_name, col_name.lower())

print(f"\nTransformed columns (lowercase):")
for col_name in policies_transformed.columns:
    print(f"    {col_name}")

print(f"\n✓ Policies data ready: {policies_transformed.count():,} rows")

# Show sample
print("\nSample data:")
display(policies_transformed.limit(5))

In [0]:
from pyspark.sql.functions import col

# Read claims.csv
claims_path = f"{config['source_volume_path']}/claims.csv"

print("Reading claims.csv...")
claims_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(claims_path)

print(f"  Rows: {claims_df.count():,}")
print(f"\nOriginal columns:")
for col_name in claims_df.columns:
    print(f"    {col_name}")

# Transform: Rename specific columns
column_mapping = {
    'age': 'driver_age',
    'date': 'incident_date',
    'hour': 'incident_hour',
    'type': 'incident_type',
    'severity': 'incident_severity'
}

claims_transformed = claims_df
for old_name, new_name in column_mapping.items():
    if old_name in claims_df.columns:
        claims_transformed = claims_transformed.withColumnRenamed(old_name, new_name)
        print(f"  Renamed: {old_name} → {new_name}")

print(f"\nTransformed columns:")
for col_name in claims_transformed.columns:
    print(f"    {col_name}")

print(f"\n✓ Claims data ready: {claims_transformed.count():,} rows")

# Show sample
print("\nSample data:")
display(claims_transformed.limit(5))

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

# Read customers.csv
customers_path = f"{config['source_volume_path']}/customers.csv"

print("Reading customers.csv...")
customers_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(customers_path)

print(f"  Rows: {customers_df.count():,}")
print(f"\nOriginal columns and types:")
for field in customers_df.schema.fields:
    print(f"    {field.name}: {field.dataType}")

# Transform: Cast customer_id to INT
customers_transformed = customers_df.withColumn("customer_id", col("customer_id").cast(IntegerType()))

print(f"\nTransformed schema:")
for field in customers_transformed.schema.fields:
    print(f"    {field.name}: {field.dataType}")

print(f"\n✓ Customers data ready: {customers_transformed.count():,} rows")

# Show sample
print("\nSample data:")
display(customers_transformed.limit(5))

## Step 2: Write Data to MySQL

Write transformed DataFrames to MySQL using JDBC with:
* **Mode**: `append` (add to existing tables)
* **Batch size**: Optimized for performance
* **Error recovery**: Each table written independently

In [0]:
def write_to_mysql(df, table_name, description=""):
    """
    Write DataFrame to MySQL table with error handling and progress tracking
    
    Args:
        df: Spark DataFrame to write
        table_name: Target MySQL table name
        description: Description for logging
    """
    import time
    
    print(f"\n{'='*80}")
    print(f"Writing to MySQL: {table_name}")
    if description:
        print(f"Description: {description}")
    print(f"{'='*80}")
    
    row_count = df.count()
    print(f"  Rows to write: {row_count:,}")
    
    if row_count == 0:
        print("  ⚠️  DataFrame is empty, skipping write")
        return False
    
    try:
        start_time = time.time()
        
        # Write to MySQL
        df.write \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", table_name) \
            .option("user", config['mysql_user']) \
            .option("password", config['mysql_password']) \
            .option("driver", "com.mysql.cj.jdbc.Driver") \
            .option("batchsize", 1000) \
            .mode("append") \
            .save()
        
        elapsed = time.time() - start_time
        rows_per_sec = row_count / elapsed if elapsed > 0 else 0
        
        print(f"  ✓ Write completed successfully")
        print(f"  Time: {elapsed:.2f} seconds")
        print(f"  Throughput: {rows_per_sec:,.0f} rows/sec")
        
        return True
        
    except Exception as e:
        print(f"  ❌ Error writing to {table_name}: {e}")
        print("\n  Troubleshooting:")
        print("    1. Verify MySQL tables exist")
        print("    2. Check MySQL user has INSERT privileges")
        print("    3. Verify column names and types match table schema")
        print(f"    4. Check MySQL logs for detailed error")
        raise

print("✓ Write function defined")

In [0]:
%pip install mysql-connector-python

print("\u2713 mysql-connector-python installed")

In [0]:
# Write policies data to MySQL using Python connector
# This approach works on serverless compute

import mysql.connector
from mysql.connector import Error

def write_dataframe_to_mysql(df, table_name, description=""):
    """
    Write Spark DataFrame to MySQL using Python mysql-connector
    Works on serverless compute (unlike Spark JDBC writer)
    """
    print(f"\n{'=' * 80}")
    print(f"Writing to MySQL: {table_name}")
    if description:
        print(f"Description: {description}")
    print(f"{'=' * 80}")
    
    # Convert DataFrame to list of dictionaries
    print("  Converting DataFrame to records...")
    rows = df.toPandas().to_dict('records')
    total_rows = len(rows)
    print(f"  Total rows to insert: {total_rows:,}")
    
    # Connect to MySQL
    print("  Connecting to MySQL...")
    try:
        connection = mysql.connector.connect(
            host=config['mysql_host'],
            port=int(config['mysql_port']),
            database=config['mysql_database'],
            user=config['mysql_user'],
            password=config['mysql_password']
        )
        
        if connection.is_connected():
            cursor = connection.cursor()
            
            # Build INSERT statement dynamically
            columns = list(rows[0].keys())
            placeholders = ', '.join(['%s'] * len(columns))
            columns_str = ', '.join([f"`{col}`" for col in columns])
            insert_query = f"INSERT INTO {table_name} ({columns_str}) VALUES ({placeholders})"
            
            print(f"  Inserting records in batches...")
            
            # Insert in batches
            batch_size = 1000
            inserted = 0
            
            for i in range(0, total_rows, batch_size):
                batch = rows[i:i+batch_size]
                batch_values = [tuple(row[col] for col in columns) for row in batch]
                
                cursor.executemany(insert_query, batch_values)
                connection.commit()
                
                inserted += len(batch)
                print(f"  Progress: {inserted:,} / {total_rows:,} rows ({inserted*100//total_rows}%)")
            
            cursor.close()
            connection.close()
            
            print(f"\n✓ SUCCESS: {inserted:,} rows written to {table_name}")
            print(f"{'=' * 80}\n")
            
    except Error as e:
        print(f"\n❌ ERROR writing to {table_name}: {e}")
        if 'connection' in locals() and connection.is_connected():
            connection.close()
        raise

# Write policies
write_dataframe_to_mysql(
    df=policies_transformed,
    table_name="policy",
    description="Insurance policy information"
)

In [0]:
# Write claims data to MySQL
write_dataframe_to_mysql(
    df=claims_transformed,
    table_name="claim",
    description="Claims data"
)

In [0]:
# Write customers data to MySQL
write_dataframe_to_mysql(
    df=customers_transformed,
    table_name="customer",
    description="Customer demographic information"
)

print("\n" + "#" * 80)
print("# ALL DATA WRITTEN TO MYSQL SUCCESSFULLY")
print("#" * 80)

## Step 3: Verify Data in MySQL

Query MySQL directly to verify data was loaded correctly.

In [0]:
# Query MySQL to verify row counts

print("Verifying data in MySQL...\n")
print("=" * 80)

tables = ['policy', 'claim', 'customer']

for table_name in tables:
    try:
        # Query row count from MySQL
        count_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", f"(SELECT COUNT(*) as row_count FROM {table_name}) as counts") \
            .option("user", config['mysql_user']) \
            .option("password", config['mysql_password']) \
            .option("driver", "com.mysql.cj.jdbc.Driver") \
            .load()
        
        count = count_df.collect()[0]['row_count']
        print(f"  {table_name:<15} {count:>10,} rows")
        
    except Exception as e:
        print(f"  {table_name:<15} ❌ Error: {e}")

print("=" * 80)
print("\n✓ Row count verification complete")

In [0]:
# Read sample data from MySQL policy table

print("Sample data from MySQL policy table:\n")

policy_sample = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "(SELECT * FROM policy LIMIT 5) as sample") \
    .option("user", config['mysql_user']) \
    .option("password", config['mysql_password']) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

display(policy_sample)

In [0]:
# Read sample data from MySQL claim table

print("Sample data from MySQL claim table:\n")

claim_sample = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "(SELECT * FROM claim LIMIT 5) as sample") \
    .option("user", config['mysql_user']) \
    .option("password", config['mysql_password']) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

display(claim_sample)

In [0]:
# Read sample data from MySQL customer table

print("Sample data from MySQL customer table:\n")

customer_sample = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "(SELECT * FROM customer LIMIT 5) as sample") \
    .option("user", config['mysql_user']) \
    .option("password", config['mysql_password']) \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .load()

display(customer_sample)

## ✓ MySQL Seeding Complete!

### What Was Done:

1. **Read CSV files** from `/Volumes/smart_claims_dev/00_landing/claimsdb`
2. **Transformed data**:
   - Policies: Lowercased column names
   - Claims: Renamed columns (age→driver_age, etc.)
   - Customers: Cast customer_id to INT
3. **Wrote to MySQL** via JDBC with batch optimization
4. **Verified data** with row counts and sample queries

### Next Steps:

1. **Test Incremental Ingestion**: Go back to the [MySQL JDBC Incremental Ingestion notebook](#notebook-1709423837384821)
2. **Run cells 2-11** to perform initial full load from MySQL to Unity Catalog
3. **Test incremental updates** by adding new records in MySQL
4. **Schedule the job** for periodic incremental refreshes

### MySQL Tables Ready:

* `policy` - Insurance policies
* `claim` - Claims data
* `customer` - Customer information

**Your MySQL database is now populated and ready for incremental ingestion testing!**